# Investigating the 2023 Regulatory Peak

2023 was the single largest year for federal regulatory docket creation in the dataset (2000-present): **50,279 dockets** and **386 million public comments**. Your job is to figure out why.

**Data sources** (pre-aggregated JSON from the Spicy Regs Dashboard):
| File | Contents |
|------|----------|
| `dockets-by-year.json` | Docket counts by year, broken down by regulatory cluster |
| `comments-by-year.json` | Public comment totals by year |
| `agency-activity.json` | Per-agency docket counts by presidential administration |
| `clusters.json` | Cluster breakdowns and percentages by administration |

All data originates from [regulations.gov](https://www.regulations.gov) via the Mirrulations ETL pipeline. The data aggregation pipeline and cluster definitions are maintained in the [Spicy Regs](https://github.com/civictechdc/spicy-regs) project.

## Resources & Guides

**Data sources & background:**
- [regulations.gov](https://www.regulations.gov) — Official U.S. federal rulemaking portal (primary data source)
- [Mirrulations project](https://github.com/MoravianUniversity/mirrern) — Open-source ETL pipeline that mirrors regulations.gov data

**Spicy Regs ecosystem:**
- [Spicy Regs](https://github.com/civictechdc/spicy-regs) — Main project: ETL pipeline and frontend for exploring federal regulatory data
- [Spicy Regs Dashboard](https://github.com/civictechdc/spicy-regs-dashboard) — Presidential regulatory timeline (this notebook's companion app)
- [Spicy Regs Agent](https://github.com/civictechdc/spicy-regs-agent) — AI-powered docket analysis agent

**Understanding the regulatory process:**
- [A Guide to the Rulemaking Process](https://www.federalregister.gov/uploads/2011/01/the_rulemaking_process.pdf) — Federal Register overview of how rules are made
- [Regulations.gov FAQ](https://www.regulations.gov/faq) — How public commenting works

**Python libraries used in this notebook:**
- [pandas documentation](https://pandas.pydata.org/docs/) — Data manipulation and analysis
- [matplotlib documentation](https://matplotlib.org/stable/contents.html) — Plotting and visualization
- [DuckDB Python API](https://duckdb.org/docs/api/python/overview) — Used in Bonus C for querying parquet files directly

## Setup

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DATA_DIR = Path("../public/data")

with open(DATA_DIR / "dockets-by-year.json") as f:
    dockets_by_year = json.load(f)
with open(DATA_DIR / "comments-by-year.json") as f:
    comments_by_year = json.load(f)
with open(DATA_DIR / "agency-activity.json") as f:
    agency_activity = json.load(f)
with open(DATA_DIR / "clusters.json") as f:
    clusters = json.load(f)

df_dockets = pd.DataFrame(dockets_by_year)
df_comments = pd.DataFrame(comments_by_year)
df_agency = pd.DataFrame(agency_activity)

CLUSTER_COLS = ['environment', 'health', 'finance', 'defense', 'transportation', 'communications', 'other']
CLUSTER_COLORS = {
    'health': '#ef4444', 'other': '#8b5cf6', 'transportation': '#3b82f6',
    'environment': '#22c55e', 'finance': '#f59e0b', 'defense': '#64748b',
    'communications': '#ec4899'
}

df_dockets.head()

## Part 1: Orientation

Start by getting a feel for the data. Here's the total docket volume over time:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['#ef4444' if y == 2023 else '#6366f1' for y in df_dockets['year']]
ax.bar(df_dockets['year'], df_dockets['total'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Total Federal Regulatory Dockets by Year', fontsize=16, fontweight='bold')
ax.set_ylabel('Number of Dockets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Shade Biden administration
ax.axvspan(2021, 2025, alpha=0.08, color='blue', label='Biden Admin')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

The 2023 bar (red) is clearly an outlier. But how much of an outlier is it, exactly?

### Exercise 1: Quantify the anomaly

Compute the following and print them:
1. The 2023 total docket count
2. The median annual docket count for the years 2005-2022 (this avoids early sparse data)
3. How many times larger 2023 is compared to that median

*Hint: use `df_dockets.loc[...]` to filter rows and `.median()` for the median.*

**Helpful resources:**
- [pandas indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) — `.loc[]`, boolean indexing
- [pandas descriptive statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics) — `.median()`, `.mean()`, `.std()`

In [ ]:
# YOUR CODE HERE


## Part 2: Cluster Decomposition

The dashboard groups agencies into clusters: `environment`, `health`, `finance`, `defense`, `transportation`, `communications`, and `other`. These cluster definitions are shared across the [Spicy Regs](https://github.com/civictechdc/spicy-regs) ecosystem.

### Exercise 2: Year-over-year growth table

Build a DataFrame comparing 2022 vs 2023 for each cluster. Include columns for `2022`, `2023`, `change`, and `pct_change`. Sort by absolute change descending. Which clusters grew the most?

*Hint: extract the 2022 and 2023 rows from `df_dockets`, then iterate over `CLUSTER_COLS`.*

**Helpful resources:**
- [pandas DataFrame construction](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) — building DataFrames from dicts
- [pandas sorting](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) — `sort_values()` with `ascending` parameter

In [ ]:
# YOUR CODE HERE


### Exercise 3: Visualize the 2023 cluster breakdown

Create a horizontal bar chart showing each cluster's docket count in 2023, colored by cluster. Use `CLUSTER_COLORS` for the colors.

*Hint: extract the 2023 row, build a small Series from `CLUSTER_COLS`, and use `ax.barh()`.*

**Helpful resources:**
- [matplotlib `barh()` — horizontal bar charts](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html)
- [matplotlib color tutorial](https://matplotlib.org/stable/users/explain/colors/colors.html) — passing hex colors and colormaps

In [ ]:
# YOUR CODE HERE


## Part 3: Health Cluster Deep Dive

The health cluster (FDA, HHS, CMS, CDC, NIH) shows one of the sharpest increases. Let's look at it over time.

### Exercise 4: Plot health dockets over time

Create a line chart of `df_dockets['health']` over `df_dockets['year']`. Highlight the 2023 data point with a larger marker or annotation. Add shaded regions for each presidential administration using `ax.axvspan()`.

Administration periods:
```
W. Bush:  2001-2009
Obama:    2009-2017
Trump:    2017-2021
Biden:    2021-2025
```

**Helpful resources:**
- [matplotlib `axvspan()` — shaded regions](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvspan.html)
- [matplotlib annotations tutorial](https://matplotlib.org/stable/users/explain/text/annotations.html) — `ax.annotate()` with arrows and offsets
- [matplotlib line plot (`plot()`)](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html) — markers, line styles

In [ ]:
# YOUR CODE HERE


## Part 4: Agency-Level Analysis

`df_agency` has per-agency docket counts for each administration (`adminId`, `agencyCode`, `agencyName`, `cluster`, `docketCount`).

### Exercise 5: Top Biden agencies

Filter `df_agency` for the Biden administration (`adminId == 'biden'`), sort by `docketCount` descending, and show the top 15 agencies. Create a horizontal bar chart colored by cluster.

Which single agency dominates? Is this surprising given what you've seen in the cluster data?

**Helpful resources:**
- [pandas `sort_values()` and `head()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) — selecting top N rows
- [pandas boolean filtering](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html) — filtering with conditions
- [regulations.gov advanced search](https://www.regulations.gov/search) — look up individual agencies and their dockets

In [ ]:
# YOUR CODE HERE


## Part 5: The Comment Spike

Docket volume is only half the story. `df_comments` tracks total public comments per year.

### Exercise 6: Visualize and quantify the comment anomaly

1. Plot `df_comments['total']` as a bar chart by year, highlighting 2023
2. Compute the ratio of 2023 comments to 2022 comments
3. What does a 386M comment year imply? Could a single docket drive most of this?

**Helpful resources:**
- [matplotlib bar chart customization](https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_colors.html) — conditional bar colors
- [Regulations.gov FAQ on public comments](https://www.regulations.gov/faq) — how mass comment campaigns work
- [pandas arithmetic operations](https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html) — computing ratios between columns/rows

In [ ]:
# YOUR CODE HERE


## Part 6: Cross-Administration Comparison

### Exercise 7: Compare presidential administrations

Using `df_dockets`, compute for each administration:
- Total dockets during their term
- Average dockets per year
- Their peak year and peak value

Build this as a DataFrame and create a bar chart of average annual docket creation by president. Color by party (D=blue, R=red).

```python
admin_years = {
    'W. Bush': range(2001, 2009),
    'Obama':   range(2009, 2017),
    'Trump':   range(2017, 2021),
    'Biden':   range(2021, 2025),
}
admin_party = {'W. Bush': 'R', 'Obama': 'D', 'Trump': 'R', 'Biden': 'D'}
```

**Helpful resources:**
- [pandas `groupby()` and aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html) — grouping and summarizing data
- [pandas `isin()` for filtering](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) — filtering rows by a list of values
- [matplotlib bar colors by category](https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_label_demo.html) — labeled, colored bar charts

In [ ]:
# YOUR CODE HERE


## Part 7: Synthesis

### Exercise 8: Write your analysis

Based on everything above, write a short summary (in markdown, in the cell below) answering: **Why was there a peak in 2023?**

Consider:
- Was it driven by one cluster or broad-based?
- Which agencies contributed most?
- What real-world policy events in 2021-2023 might explain this? (Think: legislation passed, executive orders, post-pandemic catch-up)
- Is the comment spike related to the docket spike, or a separate phenomenon?
- What can't this data tell us?

**Helpful resources:**
- [Federal Register — Executive Orders](https://www.federalregister.gov/presidential-documents/executive-orders) — search EOs by president and year
- [Congress.gov — Enacted Legislation](https://www.congress.gov/search?q=%7B%22congress%22%3A%22117%22%2C%22bill-status%22%3A%22law%22%7D) — bills signed into law during the 117th Congress (2021-2023)
- [Brookings Regulatory Tracker](https://www.brookings.edu/interactives/tracking-deregulation-in-the-trump-era/) — context on regulatory trends across administrations

*YOUR ANALYSIS HERE*


## Bonus Exercises

### Bonus A: Normalized cluster comparison

Create a stacked bar chart where each bar is normalized to 100%, showing cluster *proportions* (not counts) by year. Does 2023 look different in terms of regulatory *mix*, or just regulatory *volume*?

### Bonus B: Rolling averages

Plot a 3-year rolling average of total dockets alongside the raw annual values. Does 2023 still look anomalous in a smoothed view, or was there a building trend?

### Bonus C: Query the raw data

The underlying data lives in a Cloudflare R2 parquet file. Use DuckDB to query it directly and get a *monthly* breakdown of 2023 by agency. Which months saw the biggest spikes? The [Spicy Regs](https://github.com/civictechdc/spicy-regs) project maintains the ETL pipeline that produces this parquet file.

```python
import duckdb
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs; LOAD httpfs; SET s3_region = 'auto';")
R2_URL = "https://pub-5fc11ad134984edf8d9af452dd1849d6.r2.dev/feed_summary.parquet"
# Write your query here...
```

**Helpful resources:**
- [Spicy Regs ETL pipeline](https://github.com/civictechdc/spicy-regs) — source code for the data pipeline and parquet schema
- [matplotlib stacked bar chart](https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_stacked.html) — building normalized stacked bars
- [pandas `rolling()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) — rolling window calculations
- [DuckDB httpfs extension](https://duckdb.org/docs/extensions/httpfs/overview) — querying remote parquet files
- [DuckDB SQL reference](https://duckdb.org/docs/sql/introduction) — SQL syntax for `GROUP BY`, `DATE_PART`, etc.

In [ ]:
# YOUR CODE HERE
